# 01 Paper Fig. 2 Style EPC Contours

This notebook reproduces the Fig. 2 role from the DFTBephy Graphene paper using DeePTB-native APIs: fixed electronic `k` near `K`, a small intravalley `q` mesh around Gamma, and contour plots of acoustic-mode EPC strength. The default mesh is intentionally small so the notebook runs quickly; increase `Q_GRID_N` toward the paper-scale mesh for production checks.


In [ ]:
from pathlib import Path
import json
import numpy as np

from dptb.nn.dftbsk import DFTBSK
from dptb.postprocess.unified import TBSystem
from dptb.postprocess.unified.eph import Phonons

ROOT = Path.cwd()
DATA = ROOT / "data"
WORK = ROOT / "work"
WORK.mkdir(exist_ok=True)

SK_DATA = DATA / "skf" / "matsci-0-3"
STRUCTURE = ROOT / "graphene.vasp"
PHONOPY_YAML = DATA / "graphene" / "phonons" / "phonopy_disp.yaml"
FORCE_SETS = DATA / "graphene" / "phonons" / "FORCE_SETS"

for path, message in [
    (SK_DATA / "C-C.skf", "The bundled matsci-0-3 C-C.skf file is required."),
    (STRUCTURE, "The bundled graphene.vasp structure is required."),
    (PHONOPY_YAML, "The bundled phonopy_disp.yaml file is required."),
    (FORCE_SETS, "The bundled FORCE_SETS file is required."),
]:
    if not path.exists():
        raise FileNotFoundError(f"{path} is missing. {message}")

BASIS = {"C": ["2s", "2p"]}
model = DFTBSK(
    basis=BASIS,
    skdata=str(SK_DATA),
    overlap=True,
    dtype="float64",
    device="cpu",
    interp_method="smooth_intp",
    r_max={"C": 6.349479778742587},
)
model.eval()
system = TBSystem(data=str(STRUCTURE), calculator=model)


def regularize_tiny_negative_frequencies(phonons, tol=1e-3):
    frequencies = np.array(phonons.frequencies, copy=True)
    min_frequency = float(np.min(frequencies))
    if min_frequency < -tol:
        raise ValueError(
            f"Found phonon frequency {min_frequency} THz below tolerance {-tol} THz; "
            "this looks like a real imaginary mode, not acoustic numerical noise."
        )
    clipped = int(np.count_nonzero(frequencies < 0.0))
    frequencies[frequencies < 0.0] = 0.0
    if clipped:
        phonons = Phonons(
            qpoints=phonons.qpoints,
            frequencies=frequencies,
            eigenvectors=phonons.eigenvectors,
            masses=phonons.masses,
            cell=phonons.cell,
            scaled_positions=phonons.scaled_positions,
            metadata={
                **phonons.metadata,
                "negative_frequency_regularization": "clipped_to_zero",
                "negative_frequency_tolerance_thz": tol,
                "negative_frequency_min_before_clip_thz": min_frequency,
                "negative_frequency_clipped_count": clipped,
            },
        )
    return phonons


def get_eigenvalues_array(kpoints):
    _, evals = system.get_eigenvalues(k_points=np.asarray(kpoints, dtype=float))
    if hasattr(evals, "detach"):
        evals = evals.detach().cpu().numpy()
    return np.asarray(evals, dtype=float)


def json_default(value):
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    return str(value)

print(system.atoms)
print(system.model.name, system.model.basis)


In [ ]:
K_POINT = np.array([-2.0 / 3.0, -1.0 / 3.0, 0.0])
G_POINT = np.array([0.0, 0.0, 0.0])
CONDUCTION_BAND = 2
VALENCE_BAND = 1


def q_grid_around_gamma(n=9, qmax=0.0665):
    axis = np.linspace(-qmax, qmax, n)
    qx, qy = np.meshgrid(axis, axis, indexing="xy")
    qpoints = np.column_stack([qx.ravel(), qy.ravel(), np.zeros(qx.size)])
    return qpoints, qx, qy


def choose_k_near_k_for_energy(target_ev=0.30, nscan=41):
    alphas = np.linspace(0.0, 0.18, nscan)
    kpoints = K_POINT[None, :] + alphas[:, None] * (G_POINT - K_POINT)[None, :]
    evals = get_eigenvalues_array(kpoints)
    dirac = float(evals[0, CONDUCTION_BAND])
    energies = evals[:, CONDUCTION_BAND] - dirac
    idx = int(np.argmin(np.abs(energies - target_ev)))
    return kpoints[idx], dirac, energies[idx], evals[idx]


def acoustic_in_plane_modes(phonons, q_index):
    eig = np.asarray(phonons.eigenvectors[q_index])
    freqs = np.asarray(phonons.frequencies[q_index])
    acoustic = np.argsort(freqs)[:3]
    z_weight = np.sum(np.abs(eig[acoustic, :, 2]) ** 2, axis=1)
    in_plane = acoustic[np.argsort(z_weight)[:2]]
    ordered = in_plane[np.argsort(freqs[in_plane])]
    return {"TA": int(ordered[0]), "LA": int(ordered[1])}


def cell_area_ang2(atoms):
    cell = np.asarray(atoms.cell.array, dtype=float)
    return float(np.linalg.norm(np.cross(cell[0], cell[1])))


## Select the paper-style k and q mesh

The paper evaluates a conduction-band state close to `K`, shifted toward Gamma so the energy is about 300 meV above the Dirac point. The q mesh samples the long-wavelength intravalley region around Gamma.


In [ ]:
Q_GRID_N = 9
Q_MAX_FRACTIONAL = 0.0665
TARGET_ENERGY_EV = 0.30
DISPLACEMENT = 1e-3
BANDS = [0, 1, 2, 3]

qpoints, qx, qy = q_grid_around_gamma(Q_GRID_N, Q_MAX_FRACTIONAL)
kpoint, dirac_energy, selected_energy, selected_evals = choose_k_near_k_for_energy(TARGET_ENERGY_EV)

phonons = Phonons.from_phonopy_file(
    PHONOPY_YAML,
    qpoints=qpoints,
    force_sets_filename=str(FORCE_SETS),
)
phonons = regularize_tiny_negative_frequencies(phonons)

print("selected k", kpoint)
print("Dirac reference [eV]", dirac_energy)
print("selected conduction energy above Dirac [eV]", selected_energy)
print("qpoints", qpoints.shape)


## Compute EPC on the q contour

`coupling_strength` is `|g|^2` in the DeePTB EPC convention. For a Fig. 2 style diagnostic we plot the intraband conduction-band value as `sqrt(|g|^2) / ell_q`, with `ell_q = sqrt(hbar / (2 m_C omega_q))`, converted to Angstrom.


In [ ]:
AMU_TO_KG = 1.66053906660e-27
HBAR_J_S = 1.054571817e-34
THZ_TO_HZ = 1.0e12

nearest_nonzero_q = int(np.argmin(np.where(np.linalg.norm(qpoints[:, :2], axis=1) > 1e-12, np.linalg.norm(qpoints[:, :2], axis=1), np.inf)))
mode_map = acoustic_in_plane_modes(phonons, nearest_nonzero_q)
print("acoustic in-plane mode map", mode_map)

contour_epc = system.eph.compute_coupling(
    kpoints=kpoint[None, :],
    phonons=phonons,
    bands=BANDS,
    displacement=DISPLACEMENT,
    output_npz=WORK / "fig2_epc_contour_data.npz",
)

band_pos = BANDS.index(CONDUCTION_BAND)
strength = contour_epc.coupling_strength[:, 0, :, band_pos, band_pos]
frequency_hz = np.maximum(contour_epc.frequencies, 1e-5) * THZ_TO_HZ
mass_kg = float(np.mean(phonons.masses)) * AMU_TO_KG
ell_ang = np.sqrt(HBAR_J_S / (2.0 * mass_kg * frequency_hz)) / 1e-10
coupling_per_length = np.sqrt(np.maximum(strength, 0.0)) / ell_ang

np.savez_compressed(
    WORK / "fig2_epc_contours.npz",
    qpoints=qpoints,
    mode_indices=np.array([mode_map["TA"], mode_map["LA"]], dtype=int),
    coupling_per_length=coupling_per_length,
    kpoint=kpoint,
    dirac_energy=dirac_energy,
    selected_energy=selected_energy,
)
print(coupling_per_length.shape)


## Figure: TA/LA EPC contour

This is the DeePTB-native counterpart of paper Fig. 2. The mesh is reduced by default, but the axes, selected electronic state, acoustic-mode focus, and `|g|/ell_q` quantity match the paper intent.


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.2), dpi=140, constrained_layout=True)
levels = 12
for ax, label in zip(axes, ["TA", "LA"]):
    mode = mode_map[label]
    values = coupling_per_length[:, mode].reshape(Q_GRID_N, Q_GRID_N)
    im = ax.contourf(qx, qy, values, levels=levels, cmap="magma")
    ax.contour(qx, qy, values, levels=levels, colors="black", linewidths=0.25, alpha=0.4)
    ax.set_aspect("equal", adjustable="box")
    ax.set_title(f"{label} mode {mode}")
    ax.set_xlabel("q1 [fractional]")
    ax.set_ylabel("q2 [fractional]")
    fig.colorbar(im, ax=ax, label="|g| / ell_q [eV/Angstrom]")
fig.suptitle("Graphene EPC contour near K, intravalley q around Gamma")
fig.savefig(WORK / "figure_01_fig2_epc_contours.png", dpi=200)
plt.show()


In [ ]:
summary = {
    "figure": "paper_fig2_style_epc_contours",
    "q_grid_n": Q_GRID_N,
    "q_max_fractional": Q_MAX_FRACTIONAL,
    "target_energy_ev": TARGET_ENERGY_EV,
    "selected_energy_above_dirac_ev": float(selected_energy),
    "selected_kpoint": kpoint.tolist(),
    "mode_map": mode_map,
    "note": "Reduced-grid DeePTB-native reproduction; increase Q_GRID_N for converged publication-scale data.",
}
(WORK / "fig2_epc_contour_summary.json").write_text(json.dumps(summary, indent=2, default=json_default))
print(json.dumps(summary, indent=2, default=json_default))
